In [ ]:
# Gate G2 (spec 11, decision 23): make smoke runs the six record-size arms at 1 MiB x 1 KiB, S3 at 1 MiB and
# S4 at 64 KiB, 3 repetitions each, in under 10 minutes, on AC power with the lid open.
import json, subprocess, sys, time
from collections import Counter
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import hygiene, validate

assert hygiene.power_source() == "AC Power", f"plug the Mac in first ({hygiene.power_source()})"
assert hygiene.lid_closed() is False, "open the lid first: a closed lid sleeps the Mac despite caffeinate"
before = {p.name for p in (repo / "results/raw").glob("*_smoke_*")}
t0 = time.monotonic()
r = subprocess.run(["make", "smoke"], cwd=repo, capture_output=True, text=True)
wall_min = (time.monotonic() - t0) / 60
run_dir = next(repo / "results/raw" / n for n in {p.name for p in (repo / "results/raw").glob("*_smoke_*")} - before)
print(f"make smoke: exit {r.returncode}, {wall_min:.1f} min wall, run {run_dir.name}")
print(r.stderr[-3000:])
meta = json.loads((run_dir / "meta.json").read_text())
print({k: meta.get(k) for k in ("harness_sha", "platform", "tier")})
assert r.returncode == 0
assert wall_min < 10, f"smoke took {wall_min:.1f} min, spec 11 allows 10"

In [ ]:
# Every cell's final rows: status, row count, schema problems, and correctness of the online rows.
ledger = [json.loads(l) for l in (run_dir / "ledger.jsonl").read_text().splitlines()]
print("ledger statuses:", dict(Counter(e["status"] for e in ledger)))
bad = []
for f in sorted(run_dir.glob("*_*_*_*.jsonl")):
    rows = [json.loads(l) for l in f.read_text().splitlines()]
    probs = [str(p) for r in rows for p in validate.check_row(r, reference_platform=meta["platform"]) if p.severity == "error"]
    online = [r for r in rows if r["phase"] == "online" and r["status"] == "ok"]
    statuses = sorted({r["status"] for r in rows})
    wrong = sum(not r["correct"] for r in online)
    print(f"{f.stem:28s} {len(rows):3d} rows  {statuses}  wrong {wrong}  schema errors {len(probs)}")
    if probs or wrong or statuses != ["ok"]:
        bad.append(f.stem)
print(f"{len(bad)} cells not clean: {bad}")
assert not bad

In [ ]:
# The figures and tables from the smoke rows alone (G2). F8 to F11 need the session, network and load runs,
# so they are skipped here with a note.
r = subprocess.run(["make", "figures", f"RUNS=results/raw/{run_dir.name}"], cwd=repo, capture_output=True, text=True)
print(f"make figures: exit {r.returncode}")
print("\n".join(l for l in r.stderr.splitlines() if "skipped" in l))
made = sorted(p.name for p in (repo / "results/figures").iterdir() if p.is_file())
print(made)
need = [f"F{i}_" for i in range(1, 7)] + [f"T{i}_" for i in range(1, 5)]
missing = [n for n in need if not any(m.startswith(n) for m in made)]
print("missing:", missing)
assert r.returncode == 0 and not missing